In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Hp\\Desktop\\cifar10-image-classificatio\\research'

In [3]:
os.chdir("../")

In [4]:
import numpy as np
from pathlib import Path

In [5]:
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple


@dataclass
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path

    epochs: int
    batch_size: int
    is_augmentation: bool
    image_size: Tuple[int, int, int]

    learning_rate: float
    weight_decay: float

    rotation_range: int
    width_shift_range: float
    height_shift_range: float
    horizontal_flip: bool
    zoom_range: float
    brightness_range: tuple
    shear_range: float
    channel_shift_range: float

    reduce_lr_factor: float
    reduce_lr_patience: int
    min_learning_rate: float
    early_stopping_patience: int

In [6]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [7]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [8]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories

from pathlib import Path


class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self):

        training = self.config.training

        # ✅ FIXED KEY (prepare_model not prepare_base_model)
        prepare_model = self.config.prepare_model

        params = self.params

        create_directories([Path(training.root_dir)])

        training_data = Path("artifacts/data_ingestion")

        return TrainingConfig(

            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),

            updated_base_model_path=Path(prepare_model.model_path),

            training_data=training_data,

            epochs=int(params.EPOCHS),
            batch_size=int(params.BATCH_SIZE),
            image_size=tuple(params.IMAGE_SIZE),

            is_augmentation=str(params.AUGMENTATION).lower() == "true",

            learning_rate=float(params.LEARNING_RATE),
            weight_decay=float(params.WEIGHT_DECAY),

            rotation_range=int(params.ROTATION_RANGE),
            width_shift_range=float(params.WIDTH_SHIFT_RANGE),
            height_shift_range=float(params.HEIGHT_SHIFT_RANGE),

            horizontal_flip=str(params.HORIZONTAL_FLIP).lower() == "true",
            zoom_range=float(params.ZOOM_RANGE),
            brightness_range=tuple(params.BRIGHTNESS_RANGE),
            shear_range=float(params.SHEAR_RANGE),
            channel_shift_range=float(params.CHANNEL_SHIFT_RANGE),

            reduce_lr_factor=float(params.REDUCE_LR_FACTOR),
            reduce_lr_patience=int(params.REDUCE_LR_PATIENCE),
            min_learning_rate=float(params.MIN_LEARNING_RATE),
            early_stopping_patience=int(params.EARLY_STOPPING_PATIENCE)
        )

In [9]:
import tensorflow as tf
import numpy as np
from pathlib import Path


class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def load_data(self):

        data_path = Path(self.config.training_data)

        self.X_train = np.load(data_path / "X_train.npy")
        self.X_test = np.load(data_path / "X_test.npy")
        self.y_train = np.load(data_path / "y_train.npy")
        self.y_test = np.load(data_path / "y_test.npy")

    def get_base_model(self):

        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def get_callbacks(self):

        return [
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=self.config.reduce_lr_factor,
                patience=self.config.reduce_lr_patience,
                min_lr=self.config.min_learning_rate,
                verbose=1
            ),

            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=self.config.early_stopping_patience,
                restore_best_weights=True,
                verbose=1
            )
        ]

    def train(self):

        self.load_data()
        self.get_base_model()

        optimizer = tf.keras.optimizers.Adam(
            learning_rate=self.config.learning_rate
        )

        self.model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",  # ✅ FIX
            metrics=["accuracy"]
        )

        self.model.fit(
            self.X_train,
            self.y_train,
            validation_data=(self.X_test, self.y_test),
            epochs=self.config.epochs,
            batch_size=self.config.batch_size,
            callbacks=self.get_callbacks(),
            verbose=2
        )

        self.model.save(self.config.trained_model_path)

In [10]:
try:
    config_manager = ConfigurationManager(
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    )

    training_config = config_manager.get_training_config()

    training = Training(training_config)

    training.train()

    print("✅ Training completed successfully")

except Exception as e:
    print("❌ Error:")
    raise e

[2026-07-01 14:44:45,852: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-01 14:44:45,870: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-01 14:44:45,876: INFO: common: created directory at: artifacts]
[2026-07-01 14:44:45,880: INFO: common: created directory at: artifacts\training]
Epoch 1/10
782/782 - 830s - loss: 2.1411 - accuracy: 0.3882 - val_loss: 1.7221 - val_accuracy: 0.4582 - lr: 5.0000e-04 - 830s/epoch - 1s/step
Epoch 2/10
782/782 - 638s - loss: 1.5606 - accuracy: 0.5499 - val_loss: 1.4191 - val_accuracy: 0.5550 - lr: 5.0000e-04 - 638s/epoch - 815ms/step
Epoch 3/10
782/782 - 561s - loss: 1.3229 - accuracy: 0.6166 - val_loss: 1.0177 - val_accuracy: 0.6843 - lr: 5.0000e-04 - 561s/epoch - 717ms/step
Epoch 4/10
782/782 - 669s - loss: 1.0661 - accuracy: 0.6742 - val_loss: 1.0432 - val_accuracy: 0.6842 - lr: 5.0000e-04 - 669s/epoch - 855ms/step
Epoch 5/10
782/782 - 580s - loss: 0.9090 - accuracy: 0.7243 - val_loss: 0.8881 - val_accuracy

c:\Users\Hp\anaconda3\envs\abc\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


✅ Training completed successfully
